# SQL 练习 3：JOIN，把两张表拼起来

前两课都只用了 `sessions` 一张表。但 tag 不在这张表里，它单独存在 `session_tags` 表。

要回答「minecraft 相关的对话，哪个平台聊得最多」这种问题，就得**把两张表拼起来**，这就是 `JOIN`。

内核选 **.venv**，点中格子按 **Shift+Enter**。

In [ ]:
import duckdb

con = duckdb.connect(r"C:\Users\user\Desktop\AI analysis\Data\aiusage.duckdb", read_only=True)
print("连上了")

## 第 1 条：先看看 tag 表长什么样

In [ ]:
con.sql("SELECT * FROM session_tags LIMIT 5").df()

你会看到四列：`session_id`、`tag`、`tag_source`、`tag_type`。

**一个对话有几个 tag，就占几行**。比如前两行的 `session_id` 是同一个，一行是活动 tag `plan`，一行是主题 tag `project-ideas`。

这张表里**没有标题、没有平台**，只有 `session_id`。而 `sessions` 表里有标题和平台，也有 `session_id`。

**两张表都有 `session_id`，这就是能拼起来的依据。**

## 第 2 条：第一个 JOIN

```sql
FROM sessions AS s
JOIN session_tags AS t USING (session_id)
```

- **`AS s` / `AS t`**：给表起个短名字，后面用 `s.列名`、`t.列名` 指明这列来自哪张表
- **`USING (session_id)`**：按这一列对应。左表某行的 `session_id`，和右表哪些行相同，就拼成新的行

结果每行都同时有「对话的平台」和「它的一个 tag」。

In [ ]:
con.sql("""
    SELECT s.platform, s.user_turns, t.tag, t.tag_type
    FROM sessions AS s
    JOIN session_tags AS t USING (session_id)
    LIMIT 8
""").df()

## 第 3 条：JOIN 之后行数会变多（重要）

`sessions` 有 1054 行。拼上 tag 之后呢？

**结果是 2462 行**：因为一个对话有好几个 tag，它就被复制成了好几行。

这是 JOIN 最容易出错的地方：**JOIN 之后 `COUNT(*)` 数的不再是对话数，而是「对话 × tag」的组合数**。

要数对话数，得用 `COUNT(DISTINCT session_id)`。

In [ ]:
con.sql("""
    SELECT
        COUNT(*)                          AS 拼接后的行数,
        COUNT(DISTINCT s.session_id)      AS 实际对话数
    FROM sessions AS s
    JOIN session_tags AS t USING (session_id)
""").df()

## 第 4 条：JOIN + GROUP BY，最常用的组合

拼起来之后，就可以像前两课那样分组统计了。

下面在问：**哪些主题聊得最多？**

`WHERE t.tag_type = 'topic'` 是只看主题 tag，不看活动 tag（build、lookup 那些）。

**前几名应该是：minecraft-modding 109、ai-coding-agents 56、memes-humor 36、ai-plans-quotas 36**

In [ ]:
con.sql("""
    SELECT t.tag, COUNT(*) AS sessions
    FROM sessions AS s
    JOIN session_tags AS t USING (session_id)
    WHERE t.tag_type = 'topic'
    GROUP BY t.tag
    ORDER BY sessions DESC
    LIMIT 8
""").df()

> 这里 `COUNT(*)` 是对的：按 tag 分组之后，每组里一个对话只会出现一次。

## 第 5 条：查一个 tag 下的情况

**minecraft 模组这个主题，哪个平台聊得最多？**

这就是计划文档里「同一 tag 下三家 AI 的分配」那个分析。

**结果应该是：claude_code 42、chatgpt 39、codex 24、claude_web 4**

In [ ]:
con.sql("""
    SELECT s.platform, COUNT(*) AS sessions
    FROM sessions AS s
    JOIN session_tags AS t USING (session_id)
    WHERE t.tag = 'minecraft-modding'
    GROUP BY s.platform
    ORDER BY sessions DESC
""").df()

## 第 6 条：两张表的列一起用

JOIN 真正的好处是：**一张表的分组条件，配另一张表的数值**。

下面按活动 tag 分组（来自 `session_tags`），算平均提问次数（来自 `sessions`）。

**结果应该是：lookup 362 个对话平均 5.8 次、chat 137 个平均 13.7 次、learn 125 个平均 12.7 次、build 118 个平均 17.3 次、advice 117 个平均 20.2 次**

In [ ]:
con.sql("""
    SELECT
        t.tag                     AS activity,
        COUNT(*)                  AS sessions,
        ROUND(AVG(s.user_turns), 1) AS avg_prompts
    FROM sessions AS s
    JOIN session_tags AS t USING (session_id)
    WHERE t.tag_type = 'activity'
    GROUP BY t.tag
    ORDER BY sessions DESC
    LIMIT 5
""").df()

**读一下**：查资料（lookup）最多但最浅，平均 5.8 次就结束；
求建议（advice）只有 117 个对话，却平均聊 20 次，是最深入的。

## 第 7 条：`JOIN` 和 `LEFT JOIN` 的区别

| 写法 | 保留哪些行 |
|---|---|
| `JOIN`（内连接） | **两边都有**才保留 |
| `LEFT JOIN` | **左边全保留**，右边没有就填空值 `NULL` |

举个真实例子：`sessions_meta` 有 1060 行，但 `sessions` 只有 1054 行（有 6 个对话一条真实提问都没有，被过滤掉了）。

用 `LEFT JOIN` 保留左边全部，再筛出「右边是空」的行，就能找出这 6 个。

**`IS NULL` 是判断空值的写法**，不能写成 `= NULL`。

**结果应该是 6**

In [ ]:
con.sql("""
    SELECT COUNT(*) AS 没有真实提问的对话
    FROM sessions_meta AS m
    LEFT JOIN sessions AS s USING (session_id)
    WHERE s.session_id IS NULL
""").df()

## 自己试试

1. `flight-physics` 这个 tag 下有多少个对话？平均每个提问多少次？
2. 项目 tag（`tag_type = 'project'`）一共有哪几个？各多少个对话？
3. 难一点：把第 5 条改成看 `game-weapons` 这个 tag，并且**只看 CLI 的对话**（`s.source = 'cli'`）。
   提示：两个条件之间用 `AND` 连接。

答案在下一个格子的注释里。

In [ ]:
# 在这里写你的 SQL


# 答案：
# 1. 21 个对话，平均 49.8 次提问（这是所有主题里聊得最深的）
# 2. project:yui 8、project:vuy 4、project:simple-planes-fabric-26.2 3、
#    project:simpleplane 1、project:lct0003 1
# 3. claude_code 16、codex 5（网页版的 chatgpt 8、claude_web 1 被 source 条件排除了）